In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Set a fixed random seed so results are reproducible.
# Every time this notebook runs, the "random" numbers will be the same.
RNG = np.random.default_rng(seed=42)

# How many synthetic patients to generate
N_PATIENTS = 10_000

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Will generate {N_PATIENTS:,} patients")

numpy: 2.4.4
pandas: 3.0.2
Will generate 10,000 patients


In [2]:
# Generate patient IDs
patient_ids = np.arange(1, N_PATIENTS + 1)

# Age distribution: roughly uniform between 40 and 90
# (our cohort is adults 40+, and AF is rare below that)
ages = RNG.integers(low=40, high=91, size=N_PATIENTS)

# Sex: 51% female, 49% male — roughly matches NEL demographics
sex = RNG.choice(['F', 'M'], size=N_PATIENTS, p=[0.51, 0.49])

# Ethnicity: approximate distribution for North East London.
# NEL is one of the most ethnically diverse regions in the UK.
ethnicity_categories = ['White', 'South Asian', 'Black', 'Mixed', 'Other']
ethnicity_probs = [0.45, 0.25, 0.20, 0.05, 0.05]
ethnicity = RNG.choice(ethnicity_categories, size=N_PATIENTS, p=ethnicity_probs)

# IMD deprivation quintile: 1 = most deprived, 5 = least deprived.
# NEL skews deprived, so weight towards lower quintiles.
imd_quintile = RNG.choice([1, 2, 3, 4, 5], size=N_PATIENTS,
                          p=[0.35, 0.25, 0.20, 0.12, 0.08])

# Assemble into a DataFrame — think of this like an Excel sheet
df = pd.DataFrame({
    'patient_id': patient_ids,
    'age': ages,
    'sex': sex,
    'ethnicity': ethnicity,
    'imd_quintile': imd_quintile,
})

df.head()

,patient_id,age,sex,ethnicity,imd_quintile
0,1,44,F,White,5
1,2,79,M,White,5
2,3,73,M,White,4
3,4,62,F,Black,2
4,5,62,F,South Asian,2


In [3]:
# Check the shape (rows, columns)
print("Shape:", df.shape)

# Summary statistics for numeric columns
df.describe()

Shape: (10000, 5)


,patient_id,age,imd_quintile
count,10000.00000,10000.000000,10000.000000
mean,5000.50000,64.816000,2.328700
std,2886.89568,14.645211,1.272957
min,1.00000,40.000000,1.000000
25%,2500.75000,52.000000,1.000000
50%,5000.50000,65.000000,2.000000
75%,7500.25000,78.000000,3.000000
max,10000.00000,90.000000,5.000000


In [4]:
# Counts for categorical columns
print("Sex breakdown:")
print(df['sex'].value_counts(normalize=True))
print()
print("Ethnicity breakdown:")
print(df['ethnicity'].value_counts(normalize=True))
print()
print("IMD quintile breakdown:")
print(df['imd_quintile'].value_counts(normalize=True).sort_index())

Sex breakdown:
sex
F    0.5064
M    0.4936
Name: proportion, dtype: float64

Ethnicity breakdown:
ethnicity
White          0.4451
South Asian    0.2471
Black          0.2068
Mixed          0.0510
Other          0.0500
Name: proportion, dtype: float64

IMD quintile breakdown:
imd_quintile
1    0.3478
2    0.2499
3    0.2059
4    0.1186
5    0.0778
Name: proportion, dtype: float64


In [5]:
def sample_binary(probabilities):
    """Sample binary (0/1) outcomes given per-patient probabilities.

    Parameters
    ----------
    probabilities : array-like of floats between 0 and 1
        Each element is the probability of a 1 for that patient.

    Returns
    -------
    numpy array of 0s and 1s, same length as input.
    """
    probabilities = np.asarray(probabilities)
    return (RNG.random(size=len(probabilities)) < probabilities).astype(int)

In [6]:
# Hypertension prevalence depends on age, ethnicity, and deprivation.
# Base rate starts low, climbs with age, and is higher in certain groups.

# Start with a baseline probability for everyone
htn_logit = -4.0  # very rare at age 40 in a baseline-risk patient

# Age effect: roughly doubles per decade above 40
htn_logit = htn_logit + 0.06 * (df['age'] - 40)

# Ethnicity effect
ethnicity_htn_effect = {
    'White': 0.0,
    'South Asian': 0.3,
    'Black': 0.5,
    'Mixed': 0.1,
    'Other': 0.1,
}
htn_logit = htn_logit + df['ethnicity'].map(ethnicity_htn_effect)

# Deprivation effect (more deprived = higher risk)
htn_logit = htn_logit + 0.1 * (5 - df['imd_quintile'])

# Convert log-odds to probability using the logistic function
htn_probability = 1 / (1 + np.exp(-htn_logit))

df['hypertension'] = sample_binary(htn_probability)

print(f"Hypertension prevalence: {df['hypertension'].mean():.1%}")

Hypertension prevalence: 14.3%


In [7]:
# Diabetes risk depends on age, ethnicity, deprivation — AND hypertension
# (they share underlying risk factors and cluster together)

dm_logit = -4.5
dm_logit = dm_logit + 0.05 * (df['age'] - 40)

ethnicity_dm_effect = {
    'White': 0.0,
    'South Asian': 1.2,   # markedly elevated risk — well established in NEL populations
    'Black': 0.7,
    'Mixed': 0.3,
    'Other': 0.3,
}
dm_logit = dm_logit + df['ethnicity'].map(ethnicity_dm_effect)

dm_logit = dm_logit + 0.15 * (5 - df['imd_quintile'])

# Comorbidity clustering: hypertensive patients are more likely to have diabetes
dm_logit = dm_logit + 0.7 * df['hypertension']

dm_probability = 1 / (1 + np.exp(-dm_logit))
df['diabetes'] = sample_binary(dm_probability)

print(f"Diabetes prevalence: {df['diabetes'].mean():.1%}")
print(f"Diabetes prevalence in hypertensives: {df[df['hypertension']==1]['diabetes'].mean():.1%}")
print(f"Diabetes prevalence in non-hypertensives: {df[df['hypertension']==0]['diabetes'].mean():.1%}")

Diabetes prevalence: 12.4%
Diabetes prevalence in hypertensives: 25.7%
Diabetes prevalence in non-hypertensives: 10.2%


In [8]:
# Chronic Kidney Disease — strongly driven by diabetes, hypertension, age
ckd_logit = -5.5 + 0.08 * (df['age'] - 40) + 0.9 * df['diabetes'] + 0.7 * df['hypertension']
df['ckd'] = sample_binary(1 / (1 + np.exp(-ckd_logit)))

# Coronary Heart Disease — age, sex (male higher), diabetes, hypertension
chd_logit = (-5.5
             + 0.07 * (df['age'] - 40)
             + 0.5 * (df['sex'] == 'M').astype(int)
             + 0.6 * df['diabetes']
             + 0.5 * df['hypertension'])
df['chd'] = sample_binary(1 / (1 + np.exp(-chd_logit)))

# Heart failure — age, CHD, diabetes
hf_logit = -6.5 + 0.09 * (df['age'] - 40) + 1.2 * df['chd'] + 0.5 * df['diabetes']
df['heart_failure'] = sample_binary(1 / (1 + np.exp(-hf_logit)))

# COPD — age, deprivation (proxy for smoking exposure)
copd_logit = -5.5 + 0.07 * (df['age'] - 40) + 0.2 * (5 - df['imd_quintile'])
df['copd'] = sample_binary(1 / (1 + np.exp(-copd_logit)))

# Stroke — age, hypertension, diabetes
stroke_logit = -6.5 + 0.08 * (df['age'] - 40) + 0.9 * df['hypertension'] + 0.5 * df['diabetes']
df['prior_stroke'] = sample_binary(1 / (1 + np.exp(-stroke_logit)))

# Summary
comorbidities = ['hypertension', 'diabetes', 'ckd', 'chd', 'heart_failure', 'copd', 'prior_stroke']
print("Comorbidity prevalences:")
print(df[comorbidities].mean().apply(lambda x: f"{x:.1%}"))

Comorbidity prevalences:
hypertension     14.3%
diabetes         12.4%
ckd               7.1%
chd               5.6%
heart_failure     3.9%
copd              6.1%
prior_stroke      2.9%
dtype: str


In [9]:
# BMI: a function of age, ethnicity, deprivation, and diabetes
# Baseline mean is ~26, with variation around it
bmi_mean = (26.0
            + 0.05 * (df['age'] - 60)       # slight rise with age, centred at 60
            + 0.03 * (5 - df['imd_quintile']) * 2  # higher in more deprived
            + 2.0 * df['diabetes'])           # diabetics have higher BMI on average

# Ethnicity effect on BMI
ethnicity_bmi_effect = {
    'White': 0.0,
    'South Asian': -1.0,   # South Asians have lower BMI at same metabolic risk
    'Black': 0.8,
    'Mixed': 0.2,
    'Other': 0.0,
}
bmi_mean = bmi_mean + df['ethnicity'].map(ethnicity_bmi_effect)

# Add normally distributed noise with standard deviation 4
df['bmi'] = bmi_mean + RNG.normal(loc=0, scale=4.0, size=N_PATIENTS)

# Clip to a plausible range to avoid silly outliers
df['bmi'] = df['bmi'].clip(lower=15, upper=55).round(1)

print(df['bmi'].describe())

count    10000.000000
mean        26.589330
std          4.173381
min         15.000000
25%         23.800000
50%         26.500000
75%         29.400000
max         42.700000
Name: bmi, dtype: float64


In [10]:
# Systolic BP: depends on age, hypertension, BMI
sbp_mean = (118
            + 0.4 * (df['age'] - 40)
            + 0.3 * (df['bmi'] - 25)
            + 15 * df['hypertension'])   # hypertensives obviously higher

df['systolic_bp'] = (sbp_mean + RNG.normal(loc=0, scale=12, size=N_PATIENTS))
df['systolic_bp'] = df['systolic_bp'].clip(lower=80, upper=220).round().astype(int)

# Diastolic BP: correlated with systolic but with less spread
dbp_mean = (72
            + 0.15 * (df['age'] - 40)
            + 0.2 * (df['bmi'] - 25)
            + 8 * df['hypertension'])

df['diastolic_bp'] = (dbp_mean + RNG.normal(loc=0, scale=8, size=N_PATIENTS))
df['diastolic_bp'] = df['diastolic_bp'].clip(lower=40, upper=130).round().astype(int)

print("Systolic BP summary:")
print(df['systolic_bp'].describe())
print()
print("Mean SBP by hypertension status:")
print(df.groupby('hypertension')['systolic_bp'].mean().round(1))

Systolic BP summary:
count    10000.000000
mean       130.589800
std         15.174734
min         82.000000
25%        120.000000
50%        130.000000
75%        141.000000
max        198.000000
Name: systolic_bp, dtype: float64

Mean SBP by hypertension status:
hypertension
0    127.7
1    147.6
Name: systolic_bp, dtype: float64


In [11]:
# HbA1c (mmol/mol) — measure of long-term glucose control.
# Non-diabetics typically 30-42; diabetics well above that.

hba1c_mean = (36
              + 0.05 * (df['age'] - 40)
              + 0.2 * (df['bmi'] - 25)
              + 25 * df['diabetes'])  # large effect — defines the disease

df['hba1c'] = hba1c_mean + RNG.normal(loc=0, scale=4, size=N_PATIENTS)

# Diabetics have more variability (poorer control)
df.loc[df['diabetes'] == 1, 'hba1c'] += RNG.normal(loc=0, scale=10,
                                                     size=df['diabetes'].sum())

df['hba1c'] = df['hba1c'].clip(lower=20, upper=130).round(1)

print("HbA1c summary:")
print(df.groupby('diabetes')['hba1c'].describe().round(1))

HbA1c summary:
           count  mean   std   min   25%   50%   75%   max
diabetes                                                  
0         8758.0  37.5   4.2  20.0  34.6  37.5  40.3  55.5
1         1242.0  63.1  11.0  24.9  55.5  63.2  70.4  98.5


In [12]:
# eGFR (kidney function; lower = worse)
# Normal: 90+. CKD: <60. Very low: <30.
egfr_mean = (95
             - 0.5 * (df['age'] - 40)       # declines with age
             - 25 * df['ckd']                 # major drop if CKD
             - 3 * df['diabetes']
             - 2 * df['hypertension'])
df['egfr'] = (egfr_mean + RNG.normal(loc=0, scale=10, size=N_PATIENTS))
df['egfr'] = df['egfr'].clip(lower=5, upper=130).round().astype(int)

# Total cholesterol (mmol/L)
chol_mean = (5.0
             + 0.01 * (df['age'] - 40)
             + 0.02 * (df['bmi'] - 25)
             - 0.3 * df['chd']                # lower because they're on statins
             - 0.2 * df['diabetes'])          # likewise
df['total_cholesterol'] = chol_mean + RNG.normal(loc=0, scale=1.0, size=N_PATIENTS)
df['total_cholesterol'] = df['total_cholesterol'].clip(lower=2.5, upper=10).round(1)

# Smoking status: Never / Former / Current
# Higher current-smoking rates in more deprived groups and COPD patients
smoking_logit_current = (-2.0
                          + 0.2 * (5 - df['imd_quintile'])
                          + 1.5 * df['copd'])
p_current = 1 / (1 + np.exp(-smoking_logit_current))

# For simplicity: if not current, split remaining between Never/Former by age
p_former_given_not_current = 0.2 + 0.005 * (df['age'] - 40)

random_u = RNG.random(size=N_PATIENTS)
smoking = np.where(
    random_u < p_current, 'Current',
    np.where(random_u < p_current + (1 - p_current) * p_former_given_not_current,
             'Former', 'Never')
)
df['smoking_status'] = smoking

print("Smoking distribution:")
print(df['smoking_status'].value_counts(normalize=True).round(3))
print()
print("Mean eGFR by CKD status:")
print(df.groupby('ckd')['egfr'].mean().round(1))

Smoking distribution:
smoking_status
Never      0.529
Former     0.256
Current    0.215
Name: proportion, dtype: float64

Mean eGFR by CKD status:
ckd
0    82.6
1    48.6
Name: egfr, dtype: float64


In [13]:
print("Final shape:", df.shape)
print()
print("Columns:")
print(df.dtypes)
print()
df.head(10)

Final shape: (10000, 19)

Columns:
patient_id             int64
age                    int64
sex                      str
ethnicity                str
imd_quintile           int64
hypertension           int64
diabetes               int64
ckd                    int64
chd                    int64
heart_failure          int64
copd                   int64
prior_stroke           int64
bmi                  float64
systolic_bp            int64
diastolic_bp           int64
hba1c                float64
egfr                   int64
total_cholesterol    float64
smoking_status           str
dtype: object



,patient_id,age,sex,ethnicity,imd_quintile,hypertension,diabetes,ckd,chd,heart_failure,copd,prior_stroke,bmi,systolic_bp,diastolic_bp,hba1c,egfr,total_cholesterol,smoking_status
0,1,44,F,White,5,0,0,0,0,0,0,0,21.3,130,65,33.0,103,3.6,Current
1,2,79,M,White,5,1,0,1,0,0,0,0,25.2,160,86,41.4,60,6.5,Never
2,3,73,M,White,4,0,0,0,0,0,0,0,28.4,134,80,47.6,83,5.0,Former
3,4,62,F,Black,2,0,0,0,0,0,1,0,26.2,112,76,31.3,89,4.5,Current
4,5,62,F,South Asian,2,0,0,0,0,0,0,0,23.4,122,69,39.9,95,6.3,Former
5,6,83,F,White,2,0,0,0,0,0,0,0,31.1,129,89,39.0,72,5.0,Former
6,7,44,M,White,1,0,0,0,0,0,0,0,26.9,122,66,32.0,85,5.0,Never
7,8,75,M,Other,4,1,0,0,0,0,0,0,28.1,161,83,35.3,95,6.1,Former
8,9,50,M,White,3,0,0,0,0,0,0,0,23.9,126,80,36.8,90,5.6,Former
9,10,44,F,White,3,0,0,0,0,0,0,0,20.1,116,71,35.5,84,4.1,Never


In [14]:
# 5-year AF risk — logistic model inspired by CHARGE-AF
# The coefficients here are educational approximations, not calibrated to real data.

af_logit = -6.5  # very low baseline for a 40yo with no risk factors

# Age: dominant risk factor for AF
af_logit = af_logit + 0.08 * (df['age'] - 40)

# Sex: men higher risk
af_logit = af_logit + 0.4 * (df['sex'] == 'M').astype(int)

# Ethnicity: AF incidence is actually higher in White populations
# (a counterintuitive finding consistently seen in UK data)
ethnicity_af_effect = {
    'White': 0.3,
    'South Asian': 0.0,
    'Black': -0.2,
    'Mixed': 0.1,
    'Other': 0.0,
}
af_logit = af_logit + df['ethnicity'].map(ethnicity_af_effect)

# BMI: obesity increases risk
af_logit = af_logit + 0.02 * (df['bmi'] - 25)

# Systolic BP: elevated BP increases risk
af_logit = af_logit + 0.008 * (df['systolic_bp'] - 120)

# Comorbidities — all known AF risk factors
af_logit = af_logit + 0.5 * df['hypertension']
af_logit = af_logit + 0.4 * df['diabetes']
af_logit = af_logit + 0.8 * df['heart_failure']   # strong predictor
af_logit = af_logit + 0.5 * df['chd']
af_logit = af_logit + 0.3 * df['ckd']
af_logit = af_logit + 0.4 * df['prior_stroke']

# Smoking: current smokers higher risk
af_logit = af_logit + 0.3 * (df['smoking_status'] == 'Current').astype(int)

# Convert to probability and sample
af_probability = 1 / (1 + np.exp(-af_logit))
df['af_5yr'] = sample_binary(af_probability)

# Store the probability too — useful for debugging later
df['af_probability_true'] = af_probability.round(4)

print(f"Overall 5-year AF incidence: {df['af_5yr'].mean():.2%}")

Overall 5-year AF incidence: 5.27%


In [15]:
print("AF incidence by age band:")
df['age_band'] = pd.cut(df['age'], bins=[39, 50, 60, 70, 80, 91],
                         labels=['40-49', '50-59', '60-69', '70-79', '80-90'])
print(df.groupby('age_band', observed=True)['af_5yr'].mean().apply(lambda x: f"{x:.2%}"))
print()

print("AF incidence by comorbidity status:")
for cond in ['hypertension', 'diabetes', 'heart_failure', 'chd', 'ckd']:
    rate_yes = df[df[cond] == 1]['af_5yr'].mean()
    rate_no = df[df[cond] == 0]['af_5yr'].mean()
    print(f"  {cond:15s}: with={rate_yes:.2%}, without={rate_no:.2%}")

print()
print("AF incidence by sex:")
print(df.groupby('sex')['af_5yr'].mean().apply(lambda x: f"{x:.2%}"))

AF incidence by age band:
age_band
40-49     0.41%
50-59     0.82%
60-69     2.12%
70-79     6.31%
80-90    17.75%
Name: af_5yr, dtype: str

AF incidence by comorbidity status:
  hypertension   : with=14.64%, without=3.70%
  diabetes       : with=13.93%, without=4.04%
  heart_failure  : with=24.29%, without=4.50%
  chd            : with=20.29%, without=4.38%
  ckd            : with=17.77%, without=4.32%

AF incidence by sex:
sex
F    4.03%
M    6.54%
Name: af_5yr, dtype: str


In [16]:
# Drop the helper columns we added for inspection
df_final = df.drop(columns=['age_band', 'af_probability_true'])

print("Final dataset shape:", df_final.shape)
print()
print("Column types:")
print(df_final.dtypes)
print()
print("Any missing values?")
print(df_final.isnull().sum().sum(), "missing values total")
print()
df_final.head()

Final dataset shape: (10000, 20)

Column types:
patient_id             int64
age                    int64
sex                      str
ethnicity                str
imd_quintile           int64
hypertension           int64
diabetes               int64
ckd                    int64
chd                    int64
heart_failure          int64
copd                   int64
prior_stroke           int64
bmi                  float64
systolic_bp            int64
diastolic_bp           int64
hba1c                float64
egfr                   int64
total_cholesterol    float64
smoking_status           str
af_5yr                 int64
dtype: object

Any missing values?
0 missing values total



,patient_id,age,sex,ethnicity,imd_quintile,hypertension,diabetes,ckd,chd,heart_failure,copd,prior_stroke,bmi,systolic_bp,diastolic_bp,hba1c,egfr,total_cholesterol,smoking_status,af_5yr
0,1,44,F,White,5,0,0,0,0,0,0,0,21.3,130,65,33.0,103,3.6,Current,0
1,2,79,M,White,5,1,0,1,0,0,0,0,25.2,160,86,41.4,60,6.5,Never,0
2,3,73,M,White,4,0,0,0,0,0,0,0,28.4,134,80,47.6,83,5.0,Former,0
3,4,62,F,Black,2,0,0,0,0,0,1,0,26.2,112,76,31.3,89,4.5,Current,0
4,5,62,F,South Asian,2,0,0,0,0,0,0,0,23.4,122,69,39.9,95,6.3,Former,0


In [17]:
# Save to data/synthetic/
output_path = Path('..') / 'data' / 'synthetic' / 'eld_synthetic_v1.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(output_path, index=False)

print(f"Saved {len(df_final):,} rows to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved 10,000 rows to ../data/synthetic/eld_synthetic_v1.csv
File size: 646.0 KB
